In [1]:
##############
## INITIATE ##
##############

## Imports
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
import torch
import picklea
from PIL import Image
from torchvision import transforms
from sklearn.neighbors import KNeighborsClassifier

## Import modules
from f_wsi_reader import slide_path_to_tiles_at_coordinates
from f_wsi_reader import get_tile_map
from f_wsi_reader import get_tile_map_fast
from f_wsi_reader import read_slide
from f_wsi_reader import read_region
from f_wsi_reader import get_dimensions
from f_wsi_reader import apply_all_filters
from f_wsi_reader import apply_all_transforms
from f_wsi_reader import reassemble_tiles
from f_wsi_reader import select_patches_on_grid
from f_wsi_reader import get_tissue_mask
from f_extract_objects import extract_object_pixels
from f_extract_objects import polish
from f_extract_objects import inlay_objects
from f_extract_objects import get_bounding_box_square

#
###

In [2]:
###############
## FUNCTIONS ##
###############

## numpy_to_device
## Goal:
## Format numpy array for model.
## Inputs:
## X (list of numpy arrays): Input array for the model (B, H, W, C).
## Outputs:
## tensors (list of torch tensors): Output tensor for model (B, C, H, W).
def numpy_to_device(X, tile_size, device):
    tensors = []
    for x in X:        
        x = cv2.resize(x, (tile_size, tile_size))
        x = torch.tensor(x)        
        x = x.swapaxes(2, 1).swapaxes(1, 0) # (H, W, C) --> (C, H, W)
        x = x.float()
        x = x[None, :, :, :] # (C, H, W) --> (B=1, C, H, W)
        x = x.to(device)
        tensors += [x]        
    return tensors

#
###

In [ ]:
###########
## PATHS ##
###########

## Paths TCGA FFPE
WSI_PATHS = [
    '/Volumes/Elements/BDI/datasets/ICGCC/images/svs/TB08.0618_V5_01.svs',
    # '/Users/user/Documents/projects/GlandSeg/data/TCGA_FFPEs/TCGA-HC-7213-01Z-00-DX1.5b03a3f2-ae77-4906-81b6-e9a5c0c5ed16.svs',
    # '/Users/user/Documents/projects/GlandSeg/data/TCGA_FFPEs/TCGA-HC-A6AP-01Z-00-DX1.1E2C19B4-6757-488D-AFB4-71AE5FD6EC11.svs',
    # '/Users/user/Documents/projects/GlandSeg/data/TCGA_FFPEs/TCGA-KK-A7B2-01Z-00-DX1.3E779031-6FE4-4BD0-838C-D9ED49E1B9A7.svs',
    # '/Users/user/Documents/projects/GlandSeg/data/TCGA_FFPEs/TCGA-XJ-A83F-01Z-00-DX1.11A9B6FC-16AB-44F2-B93E-7806693D90F7.svs',
    # '/Users/user/Documents/projects/GlandSeg/data/TCGA_FFPEs/TCGA-HC-7209-01A-01-TS1.028552bb-3a9b-44dd-80c4-49d9680b2807.svs',
    # '/Users/user/Documents/projects/GlandSeg/data/TCGA_FFPEs/TCGA-HC-A6AQ-01A-01-TS1.CDFDB554-F9D7-4CB1-A4FF-48C18418AB65.svs',
    # '/Users/user/Documents/projects/GlandSeg/data/TCGA_FFPEs/TCGA-HC-A6AS-01A-01-TS1.A86A75AD-C6F2-4497-A955-393A302BE5DD.svs',
    # '/Users/user/Documents/projects/GlandSeg/data/TCGA_FFPEs/TCGA-J4-A67N-01A-01-TS1.697C38C8-0FE4-4D58-BD35-2B3510BB63D2.svs',
    # '/Users/user/Documents/projects/GlandSeg/data/TCGA_FFPEs/TCGA-SU-A7E7-01A-02-TSB.4501DDB2-75F8-46CB-A677-80981853C9FC.svs',
    # '/Users/user/Documents/projects/GlandSeg/data/TCGA_FFPEs/TCGA-YL-A8SK-01B-02-TSB.584CE622-2AE8-421A-855F-DB7CFBAE45AF.svs', # Bad quality tissue
    # '/Users/user/Documents/projects/GlandSeg/data/TCGA_FFPEs/TCGA-HC-7818-01Z-00-DX1.06895343-4b63-40b2-afbb-b31b262a4165.svs',
]

#
###

In [ ]:
###########################
## GET PATHS FROM FOLDER ##
###########################

## Path to folder
WSI_FOLDER = '/Volumes/Elements/BDI/datasets/ICGCC/images/svs/'

# Walk through WSI_FOLDER to find all .svs files
WSI_PATHS = []
for root, dirs, files in os.walk(WSI_FOLDER):
    for file in files:
        if file.lower().endswith('.svs'):
            full_path = os.path.join(root, file)
            WSI_PATHS.append(full_path)

# Optional: sort the list for consistency
WSI_PATHS.sort()

# Print result
for path in WSI_PATHS:
    print(path)

#
###

In [ ]:
################
## PARAMETERS ##
################

## Parameters for model application
LEVEL = 0 # Downsampling level 
TILE_SIZE_MODEL = 1024 # Size of tile for model
DEVICE = 'mps' # Device to load model, tiles, and perform computations
NUM_CLASSES = 3 # Number of outputs given by model

## Parameters for storing glands
BUFFER = 128 # Added distance around bounding box (in pixels)
PT_OUTPUT_FOLDER = '/Volumes/Elements/BDI/projects/GSG/outputs/o1_extracted_glands/icgcc'
# PT_OUTPUT_FOLDER = '/Users/user/Documents/projects/GlandSeg/outputs/V2/o1_extracted_glands/TCGA-HC-A6AP-01Z-00-DX1.1E2C19B4-6757-488D-AFB4-71AE5FD6EC11.svs' # Path where the gland images should be saved
if os.path.exists(PT_OUTPUT_FOLDER) == False:
    os.mkdir(PT_OUTPUT_FOLDER)

#
###

In [4]:
###########################
## LOAD GLAND MODEL DUAL ##
###########################

## Model classes
from f_model import UNet
from f_model_transformer import UNetTransformer

## Load model
device = "mps"
print(f"Using {DEVICE} device")
pt_model_in = 'models/UNet_V1_1.pth'
model_1 = UNet(n_classes=NUM_CLASSES).to(device)
if os.path.exists(pt_model_in):
    model_1.load_state_dict(torch.load(pt_model_in))
    print("Loaded model")

## Load model
device = "mps"
print(f"Using {DEVICE} device")
pt_model_in = 'models/TransUNet_V1_1.pth'
model_2 = UNetTransformer(n_classes=NUM_CLASSES).to(device)
if os.path.exists(pt_model_in):
    model_2.load_state_dict(torch.load(pt_model_in))
    print("Loaded model")

## Combine models
def model(X):
    return model_1(X) + model_2(X)

def device_to_numpy_gland(X, tile_size):
    images = []
    for x in X:
        x = torch.sigmoid(x) * 255 # Logits to probabilities to [0, 255]
        # x = (torch.sigmoid(x) > 0.75) * 255 # Logits to probabilities to [0, 255]
        # x = x * 255 # Logits to probabilities to [0, 255]
        x = x.cpu().numpy()
        x = x.swapaxes(0, 1).swapaxes(1, 2)    
        x = x.astype('uint8')
        x = cv2.resize(x, (tile_size, tile_size))
        # x = np.argmax(x, axis=2).reshape(x.shape[0], x.shape[1], 1) # Only for multiple classes
        images += [x]        
    return images

#
###

Using mps device
Loaded model
Using mps device
Loaded model


/var/folders/mf/6vvt1nwx5zz1gr1nj3hnqfzr0000gp/T/ipykernel_62578/1826005124.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_1.load_state_dict(torch.load(pt_model_

In [5]:
#############################
## LOAD NUCLEI MODEL TRUAL ##
#############################

## Load model
pt_model_in = 'models_nuclei/r13_1.pth'
model_nuclei_1 = UNet(n_classes=1).to(device)
if os.path.exists(pt_model_in):
    model_nuclei_1.load_state_dict(torch.load(pt_model_in))
    print("Loaded model")

## Load model
pt_model_in = 'models_nuclei/r13_2.pth'
model_nuclei_2 = UNet(n_classes=1).to(device)
if os.path.exists(pt_model_in):
    model_nuclei_2.load_state_dict(torch.load(pt_model_in))
    print("Loaded model")

## Load model
pt_model_in = 'models_nuclei/r13_3.pth'
model_nuclei_3 = UNet(n_classes=1).to(device)
if os.path.exists(pt_model_in):
    model_nuclei_3.load_state_dict(torch.load(pt_model_in))
    print("Loaded model")

## Combine models
def model_nuclei(X):
    return torch.sigmoid(model_nuclei_1(X)) * torch.sigmoid(model_nuclei_2(X)) * torch.sigmoid(model_nuclei_3(X))
    # return model_nuclei_1(X) + model_nuclei_2(X) + model_nuclei_3(X)

def device_to_numpy_nuclei(X, tile_size):
    images = []
    for x in X:
        # x = torch.sigmoid(x) * 255 # Logits to probabilities to [0, 255]
        # x = (torch.sigmoid(x) > 0.75) * 255 # Logits to probabilities to [0, 255]
        x = x * 255 # Logits to probabilities to [0, 255]
        # x = (x > 0.999) * 255 # Logits to probabilities to [0, 255]
        x = x.cpu().numpy()
        x = x.swapaxes(0, 1).swapaxes(1, 2)    
        x = x.astype('uint8')
        x = cv2.resize(x, (tile_size, tile_size))
        # x = np.argmax(x, axis=2).reshape(x.shape[0], x.shape[1], 1) # Only for multiple classes
        images += [x]        
    return images

#
###

Loaded model
Loaded model
Loaded model


/var/folders/mf/6vvt1nwx5zz1gr1nj3hnqfzr0000gp/T/ipykernel_62578/1092251924.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_nuclei_1.load_state_dict(torch.load(pt_

In [6]:
##############
## INITIATE ##
##############

## Debug mode
DEBUG = False

## For each WSI
for WSI_PATH in WSI_PATHS:
    print(WSI_PATH)
    
    ## Paths
    # WSI_PATH = WSI_PATHS[0] # DEBUG
    PT_INPUT = f"{PT_OUTPUT_FOLDER}/{os.path.basename(WSI_PATH)}_bboxes.npy"
    
    ## Load slide and bounding boxes
    slide = read_slide(WSI_PATH)
    bbox_list = np.load(PT_INPUT)

    ## For all bounding boxes
    for i, bbox in enumerate(bbox_list):
    
        ## Iterator
        print(f"{i+1}/{len(bbox_list)}", end='\r')
    
        ## Get bounding box of the object
        x_min, y_min, x_max, y_max = bbox # get_bounding_box_square(rescaled_gland_coords[10])
        
        ## Get location and size
        location_xy = (x_min - BUFFER, y_min - BUFFER)
        tile_size_xy = (x_max - x_min + 2 * BUFFER, y_max - y_min + 2 * BUFFER)
        # print(tile_size_xy)
        
        ## get object
        object_img = read_region(slide, location_xy, LEVEL, (np.max([TILE_SIZE_MODEL, tile_size_xy[0]]), np.max([TILE_SIZE_MODEL, tile_size_xy[1]])))
        object_img = cv2.cvtColor(object_img, cv2.COLOR_BGR2RGB)
        # print(object_img.shape)
        
        ## Glands
        with torch.no_grad():
            X = numpy_to_device([object_img], TILE_SIZE_MODEL, DEVICE)
            object_mask_gland = device_to_numpy_gland(model(X[0]),np.max([TILE_SIZE_MODEL, tile_size_xy[0]]))[0]
        
        ## Nuclei
        with torch.no_grad():
            X = numpy_to_device([object_img], TILE_SIZE_MODEL, DEVICE)
            object_mask_nuclei = device_to_numpy_nuclei(model_nuclei(X[0]),np.max([TILE_SIZE_MODEL, tile_size_xy[0]]))[0]
        
        ## Crop
        object_img = object_img[0:tile_size_xy[0],0:tile_size_xy[1]]
        object_mask_gland = object_mask_gland[0:tile_size_xy[0],0:tile_size_xy[1]]
        object_mask_nuclei = object_mask_nuclei[0:tile_size_xy[0],0:tile_size_xy[1]]
        # print(object_img.shape)
        
        ## Combine masks
        # combined_mask = (0.5 * (object_mask_gland + object_mask_nuclei[:,:,None])).astype(np.uint8)
        combined_mask = np.copy(object_mask_gland)
        #
        ## Version 1
        # combined_mask = combined_mask[:,:,[2,1,0]]
        # combined_mask[:,:,0] = object_mask_nuclei
        #
        ## Version 2
        combined_mask = combined_mask[:,:,[2,1,0]]
        # combined_mask[:,:,0] = object_mask_nuclei
        combined_mask[:,:,2] = object_mask_nuclei
        
        ## Save the object image    
        object_filename = os.path.join(PT_OUTPUT_FOLDER, f'{os.path.basename(WSI_PATH)}__{x_min}_{x_max}_{y_min}_{y_max}_raw.png')
        cv2.imwrite(object_filename, object_img)
        
        ## Save the object image    
        object_filename = os.path.join(PT_OUTPUT_FOLDER, f'{os.path.basename(WSI_PATH)}__{x_min}_{x_max}_{y_min}_{y_max}_mask_glands.png')
        cv2.imwrite(object_filename, object_mask_gland)
        
        ## Save the object image    
        object_filename = os.path.join(PT_OUTPUT_FOLDER, f'{os.path.basename(WSI_PATH)}__{x_min}_{x_max}_{y_min}_{y_max}_mask_nuclei.png')
        cv2.imwrite(object_filename, object_mask_nuclei)
        
        ## Save the object image    
        object_filename = os.path.join(PT_OUTPUT_FOLDER, f'{os.path.basename(WSI_PATH)}__{x_min}_{x_max}_{y_min}_{y_max}_mask_both.png')
        cv2.imwrite(object_filename, combined_mask)
    
        if DEBUG:
            break
    
    #
    ###

#
###

/Volumes/Elements/BDI/datasets/ICGCC/images/svs/TB08.0618_V5_01.svs
125/125

In [9]:
"""
#########################
## PROCESS ALL OBJECTS ##
#########################

## Debug mode
DEBUG = False

## For all bounding boxes
for i, bbox in enumerate(bbox_list):

    ## Iterator
    print(f"{i+1}/{len(bbox_list)}", end='\r')

    ## Get bounding box of the object
    x_min, y_min, x_max, y_max = bbox # get_bounding_box_square(rescaled_gland_coords[10])
    
    ## Get location and size
    location_xy = (x_min - BUFFER, y_min - BUFFER)
    tile_size_xy = (x_max - x_min + 2 * BUFFER, y_max - y_min + 2 * BUFFER)
    # print(tile_size_xy)
    
    ## get object
    object_img = read_region(slide, location_xy, LEVEL, (np.max([TILE_SIZE_MODEL, tile_size_xy[0]]), np.max([TILE_SIZE_MODEL, tile_size_xy[1]])))
    object_img = cv2.cvtColor(object_img, cv2.COLOR_BGR2RGB)
    # print(object_img.shape)
    
    ## Glands
    with torch.no_grad():
        X = numpy_to_device([object_img], TILE_SIZE_MODEL, DEVICE)
        object_mask_gland = device_to_numpy_gland(model(X[0]),np.max([TILE_SIZE_MODEL, tile_size_xy[0]]))[0]
    
    ## Nuclei
    with torch.no_grad():
        X = numpy_to_device([object_img], TILE_SIZE_MODEL, DEVICE)
        object_mask_nuclei = device_to_numpy_nuclei(model_nuclei(X[0]),np.max([TILE_SIZE_MODEL, tile_size_xy[0]]))[0]
    
    ## Crop
    object_img = object_img[0:tile_size_xy[0],0:tile_size_xy[1]]
    object_mask_gland = object_mask_gland[0:tile_size_xy[0],0:tile_size_xy[1]]
    object_mask_nuclei = object_mask_nuclei[0:tile_size_xy[0],0:tile_size_xy[1]]
    # print(object_img.shape)
    
    ## Combine masks
    # combined_mask = (0.5 * (object_mask_gland + object_mask_nuclei[:,:,None])).astype(np.uint8)
    combined_mask = np.copy(object_mask_gland)
    #
    ## Version 1
    # combined_mask = combined_mask[:,:,[2,1,0]]
    # combined_mask[:,:,0] = object_mask_nuclei
    #
    ## Version 2
    combined_mask = combined_mask[:,:,[2,1,0]]
    # combined_mask[:,:,0] = object_mask_nuclei
    combined_mask[:,:,2] = object_mask_nuclei
    
    ## Save the object image    
    object_filename = os.path.join(PT_OUTPUT_FOLDER, f'{os.path.basename(WSI_PATH)}__{x_min}_{x_max}_{y_min}_{y_max}_raw.png')
    cv2.imwrite(object_filename, object_img)
    
    ## Save the object image    
    object_filename = os.path.join(PT_OUTPUT_FOLDER, f'{os.path.basename(WSI_PATH)}__{x_min}_{x_max}_{y_min}_{y_max}_mask_glands.png')
    cv2.imwrite(object_filename, object_mask_gland)
    
    ## Save the object image    
    object_filename = os.path.join(PT_OUTPUT_FOLDER, f'{os.path.basename(WSI_PATH)}__{x_min}_{x_max}_{y_min}_{y_max}_mask_nuclei.png')
    cv2.imwrite(object_filename, object_mask_nuclei)
    
    ## Save the object image    
    object_filename = os.path.join(PT_OUTPUT_FOLDER, f'{os.path.basename(WSI_PATH)}__{x_min}_{x_max}_{y_min}_{y_max}_mask_both.png')
    cv2.imwrite(object_filename, combined_mask)

    if DEBUG:
        break

#
###
"""

4659/4659